In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

SILVER_ORDER_ITEMS = f"{CATALOG}.{SCHEMA_PREFIX}_silver.order_items"
SILVER_ORDERS = f"{CATALOG}.{SCHEMA_PREFIX}_silver.orders"
DIM_CUSTOMER = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_customer"
FACT_ORDER_ITEMS = f"{CATALOG}.{SCHEMA_PREFIX}_gold.fact_order_items"

FACT_COLUMNS = [
    "order_id", "line_number", "date_key", "customer_sk", "product_id",
    "country_key", "order_channel_key", "quantity", "unit_price_usd",
    "discount_pct", "gross_amount_usd", "discount_amount_usd", "net_amount_usd", "processed_at",
]

DIM_DATE = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_date"
DIM_COUNTRY = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_country"
DIM_ORDER_CHANNEL = f"{CATALOG}.{SCHEMA_PREFIX}_gold.dim_order_channel"

UNKNOWN_MEMBER = "Unknown"

In [0]:
fact_df = spark.sql(f"""
WITH lines AS (
    SELECT
        i.order_id,
        i.line_number,
        i.product_id,
        i.quantity,
        i.unit_price_usd,
        i.line_total_usd,
        o.customer_id,
        o.order_time,
        o.country,
        o.payment_method,
        o.device,
        o.source,
        o.discount_pct,
        CAST(o.total_usd AS DECIMAL(12,2))                                     AS order_total_usd,
        CAST(i.line_total_usd * (1 - o.discount_pct / 100) AS DECIMAL(12,2))   AS net_rounded_usd
    FROM {SILVER_ORDER_ITEMS} i
    JOIN {SILVER_ORDERS} o ON i.order_id = o.order_id
),
allocated AS (
    -- Rounding each line separately can leave the order a few cents off its total;
    -- the residual goes to the last line so line amounts reconcile exactly with the order.
    SELECT
        *,
        CAST(
            net_rounded_usd
            + CASE
                WHEN line_number = MAX(line_number) OVER (PARTITION BY order_id)
                THEN order_total_usd - SUM(net_rounded_usd) OVER (PARTITION BY order_id)
                ELSE 0
              END
        AS DECIMAL(12,2)) AS net_amount_usd
    FROM lines
)
SELECT
    l.order_id,
    l.line_number,
    CAST(date_format(l.order_time, 'yyyyMMdd') AS INT)         AS date_key,
    c.customer_sk,
    l.product_id,
    co.country_key,
    ch.order_channel_key,
    l.quantity,
    CAST(l.unit_price_usd AS DECIMAL(12,2))                    AS unit_price_usd,
    l.discount_pct,
    CAST(l.line_total_usd AS DECIMAL(12,2))                    AS gross_amount_usd,
    CAST(l.line_total_usd AS DECIMAL(12,2)) - l.net_amount_usd AS discount_amount_usd,
    l.net_amount_usd,
    current_timestamp()                                        AS processed_at
FROM allocated l
LEFT JOIN {DIM_CUSTOMER} c
    ON  l.customer_id = c.customer_id
    AND l.order_time BETWEEN c.effective_from AND c.effective_to
LEFT JOIN {DIM_COUNTRY} co
    ON  co.country_code = COALESCE(l.country, '{UNKNOWN_MEMBER}')
LEFT JOIN {DIM_ORDER_CHANNEL} ch
    ON  ch.payment_method = COALESCE(l.payment_method, '{UNKNOWN_MEMBER}')
    AND ch.device         = COALESCE(l.device,         '{UNKNOWN_MEMBER}')
    AND ch.source         = COALESCE(l.source,         '{UNKNOWN_MEMBER}')
""")

In [0]:
expected_rows = spark.table(SILVER_ORDER_ITEMS).count()

stats = fact_df.agg(
    F.count("*").alias("rows"),
    F.count_if(F.col("customer_sk").isNull()).alias("lines_without_customer"),
    F.count_if(F.col("country_key").isNull() | F.col("order_channel_key").isNull()).alias("lines_without_dimension_key"),
).first()

if stats["rows"] != expected_rows:
    raise ValueError(
        f"Fact row count {stats['rows']} differs from silver {expected_rows}: "
        "customer versions likely overlap and multiply order lines"
    )
if stats["lines_without_customer"] > 0:
    raise ValueError(
        f"{stats['lines_without_customer']} order lines have no customer version valid at order time"
    )
if stats["lines_without_dimension_key"] > 0:
    raise ValueError(
        f"{stats['lines_without_dimension_key']} order lines have no country or order channel key; "
        "run 05_gold_dims before the fact"
    )

lines_outside_calendar = (
    fact_df.join(spark.table(DIM_DATE).select("date_key"), "date_key", "left_anti").count()
)
if lines_outside_calendar > 0:
    raise ValueError(
        f"{lines_outside_calendar} order lines fall outside the dim_date range; "
        "extend dim_date by re-running it with a later end_date"
    )

In [0]:
fact_df.select(*FACT_COLUMNS).createOrReplaceTempView("fact_order_items_source")

spark.sql(f"""
INSERT OVERWRITE {FACT_ORDER_ITEMS} ({", ".join(FACT_COLUMNS)})
SELECT {", ".join(FACT_COLUMNS)} FROM fact_order_items_source
""")